# StudyMate — Your Personal Research & Homework Assistant

**A complete single-agent project: build it, test it, deploy it as a real app.**

## What we're building

StudyMate is an AI agent with three abilities working together:

1. **Web search** — for current facts, definitions, and general knowledge.
2. **Calculator** — for anything numeric (grades, unit conversions, statistics).
3. **"My Notes" search (RAG)** — you upload your own documents (class notes, a textbook chapter, past exam questions, a school policy PDF) and the agent searches *those specific files* by meaning, not keyword, before falling back to the open web.

It also **remembers the conversation**, so you can ask follow-up questions naturally.

### Why this is genuinely useful, not a toy
- A student revising for exams can upload their own notes and ask "explain photosynthesis the way my teacher wrote it," and get an answer grounded in their actual material, not a generic internet answer.
- A teacher can upload a syllabus or rubric and ask the agent to check whether a lesson plan covers all required outcomes.
- Anyone can upload a contract, receipt, or policy document and ask plain-English questions about it instead of reading the whole thing.
- Unlike a plain chatbot, it doesn't have to guess — it looks things up, in your documents or on the web, before answering.

### What you'll end up with
By the end of this notebook you will have:
- A working agent, testable right here in Colab
- A real deployable app (`app.py`) with a chat interface AND a file-upload box
- A public URL you can send to anyone


## Setup

Free API keys needed:
- **Groq** (the agent's brain): https://console.groq.com/keys
- **Tavily** (web search, free tier): https://tavily.com/

No other paid services required.


In [1]:
!pip install -q groq tavily-python chromadb sentence-transformers pypdf streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

import groq
from tavily import TavilyClient

client = groq.Groq()
tavily = TavilyClient()

# Groq-hosted model used for the agent's reasoning + tool calling.
# openai/gpt-oss-120b is Groq's current recommended model for strong tool-use quality.
MODEL_NAME = "openai/gpt-oss-120b"

print("Ready.")

Ready.


## Step 1 — Let the agent read your documents (the RAG pipeline)

Upload a PDF or text file. We'll break it into chunks, turn each chunk into a searchable "embedding," and store them so the agent can pull out the most relevant pieces later — this is exactly the RAG pattern from Week 4, applied to a real file instead of sample text.


In [3]:
from google.colab import files
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions

uploaded = files.upload()  # pick a PDF or .txt file from your computer

def extract_text(filename):
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    else:
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

def chunk_text(text, chunk_size=800, overlap=100):
    # simple fixed-size chunking with a bit of overlap so we don't cut a sentence in half
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]

filename = list(uploaded.keys())[0]
raw_text = extract_text(filename)
chunks = chunk_text(raw_text)
print(f"Extracted {len(raw_text)} characters, split into {len(chunks)} chunks.")
print("\nFirst chunk preview:\n", chunks[0][:300])


Saving A Survey of Large Language Models.pdf to A Survey of Large Language Models.pdf
Extracted 259255 characters, split into 371 chunks.

First chunk preview:
 A Survey of Large Language Models
Wayne Xin Zhao1, Kun Zhou2*, Junyi Li1*, Tianyi Tang1, Zican Dong1, Yupeng Hou1, Beichen Zhang1, Yingqian Min1,
Junjie Zhang1, Peiyu Liu1, Xiaolei Wang1, Yifan Du1, Chen Yang1, Yushuo Chen1, Zhipeng Chen1, Jinhao Jiang1,
Ruiyang Ren1, Yifan Li1, Xinyu Tang1, Zikang 


In [4]:
chroma_client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(name="my_notes", embedding_function=embed_fn)
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Indexed {len(chunks)} chunks from '{filename}' into the vector store.")

# quick manual test — search by MEANING, not exact words
test_query = "What is this document mainly about?"
results = collection.query(query_texts=[test_query], n_results=2)
for doc in results["documents"][0]:
    print("\n---\n", doc[:300])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 371 chunks from 'A Survey of Large Language Models.pdf' into the vector store.

---
 diverse  mixture  of  text
datasets as the pre-training corpus. These datasets generally fall into
the following three categories.
General  text  data,  such  as  webpages  and  books,  is  utilized  by
most  LLMs  [13,15]  due  to  its  large-scale,  diverse,  and  accessible
nature.  In  particula

---
 i  A  S. Position:  key  claims  in  LLM  research
have  a  long  tail  of  footnotes.  In: Proceedings  of  the  41st  International
Conference on Machine Learning. 2024, 1735
 [34]   Penedo G, Kydlíček H, Allal L B, Lozhkov A, Mitchell M, Raffel
C A, Von Werra L, Wolf T. The FineWeb datasets: deca


## Step 2 — Build the agent's three tools

Same pattern as Week 4: a plain Python function, plus a one-sentence description, for each ability.


In [5]:
def search_my_notes(query: str) -> str:
    # Search the user's own uploaded document by meaning.
    results = collection.query(query_texts=[query], n_results=3)
    if not results["documents"][0]:
        return "No relevant content found in the uploaded document."
    return "\n\n".join(results["documents"][0])

def web_search(query: str) -> str:
    # Search the open web for current facts, definitions, or general knowledge.
    results = tavily.search(query=query, max_results=3)
    return "\n".join(f"- {r['title']}: {r['content'][:200]}" for r in results["results"])

def calculator(expression: str) -> str:
    # Evaluate a mathematical expression, e.g. grade averages, percentages, conversions.
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Groq's API is OpenAI-compatible, so tools are described in OpenAI's
# "function calling" schema: each tool is wrapped in a {"type": "function", "function": {...}}
# object, and the parameters use JSON Schema (same idea as Anthropic's input_schema,
# just nested one level deeper).
study_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_my_notes",
            "description": "Search the user's own uploaded document/notes for relevant content. ALWAYS try this first for anything that could be covered in the user's material before searching the open web.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "What to look for in the notes."}},
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the open web for current facts, definitions, or general knowledge not found in the user's own notes.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "The search query."}},
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression, e.g. for grade averages, percentages, or unit conversions.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "A Python-evaluable math expression."}},
                "required": ["expression"]
            }
        }
    }
]

def run_tool(name, tool_input):
    if name == "search_my_notes":
        return search_my_notes(**tool_input)
    elif name == "web_search":
        return web_search(**tool_input)
    elif name == "calculator":
        return calculator(**tool_input)
    return f"Error: tool '{name}' does not exist."

print("Tools ready: search_my_notes, web_search, calculator")


Tools ready: search_my_notes, web_search, calculator


## Step 3 — The agent loop, with memory

This is the same ReAct loop from Week 4, wired to keep conversation history so follow-up questions work naturally.


In [6]:
import json

SYSTEM_PROMPT = (
    "You are StudyMate, a helpful research and homework assistant. "
    "Always check the user's own notes (search_my_notes) before searching the open web, "
    "when the question could plausibly be covered in their uploaded document. "
    "Be clear and concise. If neither tool has the answer, say so honestly instead of guessing."
)

# In the OpenAI/Groq message format, the system prompt is just the first message
# in the list (role="system"), rather than a separate `system=` argument.
conversation = [{"role": "system", "content": SYSTEM_PROMPT}]  # this list IS the agent's memory

def studymate_ask(question, max_iterations=6, verbose=True):
    conversation.append({"role": "user", "content": question})

    for step in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=1024,
            tools=study_tools,
            messages=conversation
        )
        message = response.choices[0].message
        # Groq's SDK wants the assistant message appended as a plain dict
        conversation.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"[calling {tool_name}({tool_input})]")
            result = run_tool(tool_name, tool_input)
            conversation.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    return "Sorry, I couldn't finish that in time."

# Try it — ask something that should come from YOUR uploaded document
print(studymate_ask("What is this document about, in simple terms?"))


[calling search_my_notes({'query': ''})]
**Simple summary:**  
The document is a survey of recent research on large language models (LLMs). It talks about three main things:

1. **How we test and compare LLMs** – new benchmarks and platforms (e.g., Chatbot Arena, GPQA, math Olympiad‑style tests) that measure how well models reason, follow instructions, or answer tough questions.  

2. **How we improve LLMs after the big “pre‑training” step** – especially using **reinforcement learning (RL)** so the model learns to produce answers that match human preferences or specific goals, rather than just predicting the next word.  

3. **How we make LLMs smaller and faster** – techniques like **pruning** (removing unnecessary parts) and **quantization/compression** that keep the model’s accuracy while cutting its size and compute cost.

In plain language, the paper explains the latest ways researchers check how good language models are, fine‑tune them to behave better using feedback, and shrink t

### Test drive — try a real conversation

Ask a follow-up (memory should kick in), then something clearly NOT in your document (web search should kick in), then a calculation.


In [7]:
print(studymate_ask("Can you explain that in more detail?"))          # tests memory
print(studymate_ask("What year is it currently?"))                    # tests web_search
print(studymate_ask("If I scored 68, 74, and 81 on three tests, what's my average?"))  # tests calculator


Below is a more‑in‑depth, yet still plain‑English, walk‑through of the three big topics the paper covers.

---

## 1. How we **evaluate** (test) large language models  

| What the paper talks about | What it means in everyday language |
|----------------------------|-------------------------------------|
| **Human‑preference platforms** (e.g., *Chatbot Arena*) | Researchers let real people compare two model answers side‑by‑side and pick the better one. The “winner” model is the one humans like more. |
| **Graduate‑level Q&A benchmarks** (e.g., *GPQA*) | A test set of really hard questions—similar to those you’d see on a graduate exam or a Google interview. If a model can answer many of these correctly, it shows deep knowledge. |
| **Olympiad‑style math benchmarks** (e.g., the 2025 “math Olympiad” dataset) | Problems that require multi‑step reasoning, symbolic manipulation, and creativity—much tougher than the typical “solve‑a‑simple‑equation” tasks. Success here shows the model can re

## Step 4 — Deploy it as a real app

This turns StudyMate into a chat website with a file-upload box, that anyone can use — no Colab, no code visible, just a link.

**This part runs outside Colab.** Steps:
1. Run the next cell — it writes `app.py` and `requirements.txt` to your Colab files.
2. Download both (folder icon on the left sidebar → right-click → Download).
3. Create a free GitHub account and a new repository. Upload both files to it.
4. Go to https://streamlit.io/cloud → sign in with GitHub → "New app" → select your repository and `app.py`.
5. In the app's Settings → Secrets, add:
   ```
   GROQ_API_KEY = "your key here"
   TAVILY_API_KEY = "your key here"
   ```
6. Click Deploy. In a minute or two you get a public URL like `https://studymate-yourname.streamlit.app` — send it to anyone.


In [8]:
%%writefile app.py
"""
StudyMate — Personal Research & Homework Assistant
A single-agent Streamlit app: web search + calculator + RAG over your own uploaded document,
with conversation memory. Deploy this file directly on Streamlit Cloud.

Setup on Streamlit Cloud:
1. Push this file and requirements.txt to a GitHub repo.
2. On streamlit.io/cloud, create a new app pointing at this file.
3. In the app's Settings -> Secrets, add:
     GROQ_API_KEY = "..."
     TAVILY_API_KEY = "..."
"""

import json
import streamlit as st
import groq
from tavily import TavilyClient
import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader
import io

st.set_page_config(page_title="StudyMate", page_icon="📚", layout="centered")
st.title("📚 StudyMate")
st.caption("Your research & homework assistant — upload your notes, then ask anything.")

# ---------- clients (built once, using secrets) ----------
client = groq.Groq(api_key=st.secrets["GROQ_API_KEY"])
tavily = TavilyClient(api_key=st.secrets["TAVILY_API_KEY"])

# Groq-hosted model used for the agent's reasoning + tool calling.
# openai/gpt-oss-120b is Groq's current recommended model for strong tool-use quality.
MODEL_NAME = "openai/gpt-oss-120b"

SYSTEM_PROMPT = (
    "You are StudyMate, a helpful research and homework assistant. "
    "Always check the user's own notes (search_my_notes) before searching the open web, "
    "when the question could plausibly be covered in their uploaded document. "
    "Be clear and concise. If neither tool has the answer, say so honestly instead of guessing."
)


@st.cache_resource
def get_collection():
    chroma_client = chromadb.Client()
    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    return chroma_client.get_or_create_collection(name="my_notes", embedding_function=embed_fn)


collection = get_collection()

# ---------- session state ----------
# The conversation list IS the agent's memory, and its first entry is the
# system prompt (Groq/OpenAI-style messages put the system prompt in the
# messages list itself, rather than passing it as a separate argument).
if "conversation" not in st.session_state:
    st.session_state.conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
if "doc_indexed" not in st.session_state:
    st.session_state.doc_indexed = False


# ---------- document ingestion ----------
def extract_text(uploaded_file):
    if uploaded_file.name.lower().endswith(".pdf"):
        reader = PdfReader(io.BytesIO(uploaded_file.read()))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    return uploaded_file.read().decode("utf-8", errors="ignore")


def chunk_text(text, chunk_size=800, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]


with st.sidebar:
    st.header("Your documents")
    uploaded_file = st.file_uploader("Upload notes (PDF or .txt)", type=["pdf", "txt"])
    if uploaded_file is not None and st.button("Index this document"):
        with st.spinner("Reading and indexing..."):
            text = extract_text(uploaded_file)
            chunks = chunk_text(text)
            existing = collection.count()
            collection.add(
                documents=chunks,
                ids=[f"chunk_{existing + i}" for i in range(len(chunks))],
            )
            st.session_state.doc_indexed = True
        st.success(f"Indexed {len(chunks)} chunks from {uploaded_file.name}.")
    if st.session_state.doc_indexed:
        st.info("StudyMate will check your notes first before searching the web.")
    if st.button("Clear conversation"):
        st.session_state.conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
        st.rerun()

# ---------- tools ----------
def search_my_notes(query: str) -> str:
    if collection.count() == 0:
        return "No documents have been uploaded yet."
    results = collection.query(query_texts=[query], n_results=3)
    if not results["documents"][0]:
        return "No relevant content found in the uploaded document."
    return "\n\n".join(results["documents"][0])


def web_search(query: str) -> str:
    results = tavily.search(query=query, max_results=3)
    return "\n".join(f"- {r['title']}: {r['content'][:200]}" for r in results["results"])


def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error evaluating expression: {e}"


# Groq's API is OpenAI-compatible, so tools are described in OpenAI's
# "function calling" schema: each tool is wrapped in a {"type": "function", "function": {...}}
# object, with parameters as JSON Schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_my_notes",
            "description": "Search the user's own uploaded document/notes for relevant content. ALWAYS try this first for anything that could be covered in the user's material before searching the open web.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "What to look for in the notes."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the open web for current facts, definitions, or general knowledge not found in the user's own notes.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "The search query."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression, e.g. for grade averages, percentages, or unit conversions.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "A Python-evaluable math expression."}},
                "required": ["expression"],
            },
        },
    },
]


def run_tool(name, tool_input):
    if name == "search_my_notes":
        return search_my_notes(**tool_input)
    elif name == "web_search":
        return web_search(**tool_input)
    elif name == "calculator":
        return calculator(**tool_input)
    return f"Error: tool '{name}' does not exist."


def run_agent(messages, max_iterations=6):
    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )
        message = response.choices[0].message
        # Groq's SDK wants the assistant message appended as a plain dict.
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return message.content, messages

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            result = run_tool(tool_name, tool_input)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })
    return "Sorry, I couldn't finish that in time.", messages


# ---------- chat UI ----------
for msg in st.session_state.conversation:
    if msg["role"] in ("user", "assistant") and isinstance(msg.get("content"), str) and msg["content"]:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

user_input = st.chat_input("Ask about your notes, or anything else...")
if user_input:
    st.session_state.conversation.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer, st.session_state.conversation = run_agent(st.session_state.conversation)
            st.write(answer)


Writing app.py


In [9]:
%%writefile requirements.txt
streamlit
groq
tavily-python
chromadb
sentence-transformers
pypdf


Writing requirements.txt


### Confirm your 3 files are ready

Run the two cells above, then check the Colab file browser (folder icon, left sidebar) — you should now see:

- `app.py`
- `requirements.txt`

alongside this notebook. Nothing was created outside Colab; both files were written to disk by the `%%writefile` cells you just ran, using the exact same tools, agent loop, and system prompt you already tested above.

**To deploy from here:**
1. Download both files (folder icon → right-click each → Download), or connect this Colab session to GitHub directly if you prefer.
2. Create a free GitHub repository and upload `app.py` and `requirements.txt` to it.
3. Go to https://streamlit.io/cloud → sign in with GitHub → "New app" → point it at your repo and `app.py`.
4. In Settings → Secrets, add your `GROQ_API_KEY` and `TAVILY_API_KEY`.
5. Click Deploy — you'll get a public URL in about a minute.


### Ideas to extend it further
- Add a 4th tool: a **flashcard generator** that turns notes into Q&A pairs.
- Let it index **multiple** documents and tell the user which file an answer came from.
- Add a **grade calculator** mode with a stricter system prompt for a report-writing assistant like the ones used for student assessments.
- Try swapping `MODEL_NAME` for a different Groq-hosted model (e.g. a smaller/faster one) to compare speed and quality on the same questions.
